# PDF → Chunks → Embeddings → ChromaDB  
###  LangChain Pipeline · HuggingFace · ChromaDB · No API Key Needed

| Component | Library |
|---|---|
| PDF Loader | `LangChain PyMuPDFLoader` |
| Text Splitter | `LangChain RecursiveCharacterTextSplitter` |
| Embeddings | `LangChain HuggingFaceEmbeddings (sentence-transformers/all-MiniLM-L6-v2)` |
| Vector Store | `LangChain Chroma` |
| Retriever | `LangChain VectorStoreRetriever` |

##  Install Dependencies

In [3]:
!pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma pymupdf chromadb sentence-transformers

## Imports

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print(' All LangChain components imported successfully!')

 All LangChain components imported successfully!


##  Configuration

In [5]:
# ── SETTINGS ──────────────────────────────────────────────────────────────────
PDF_PATH        = "sample_knowledge.txt"   # Path to your PDF
CHROMA_DIR      = "./chroma_db"         #  Local folder to persist ChromaDB
COLLECTION_NAME = "my_pdf_collection"   #  Collection name inside ChromaDB

# Chunking
CHUNK_SIZE      = 100    # characters per chunk
CHUNK_OVERLAP   = 20     # overlap between consecutive chunks

# HuggingFace model — free, runs fully locally
# Alternatives:
#   'BAAI/bge-small-en-v1.5'   → faster & lighter
#   'BAAI/bge-large-en-v1.5'   → higher accuracy
#   'paraphrase-MiniLM-L6-v2'  → very lightweight
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
# ──────────────────────────────────────────────────────────────────────────────

print(f' PDF          : {PDF_PATH}')
print(f' Chunk size   : {CHUNK_SIZE} | Overlap: {CHUNK_OVERLAP}')
print(f' Embeddings   : {EMBEDDING_MODEL}')
print(f'  ChromaDB dir : {CHROMA_DIR}')

 PDF          : sample_knowledge.txt
 Chunk size   : 100 | Overlap: 20
 Embeddings   : paraphrase-MiniLM-L3-v2
  ChromaDB dir : ./chroma_db


##  Step 1 — Load PDF with LangChain

In [6]:
# PyMuPDFLoader loads each page as a separate LangChain Document
# Each Document has: .page_content (text) + .metadata (source, page number)

loader = PyMuPDFLoader(PDF_PATH)
pages  = loader.load()

print(f' Loaded {len(pages)} pages from "{PDF_PATH}"')
print(f'\n Sample Page [0] metadata : {pages[0].metadata}')
print(f'\n Sample Page [0] content  :\n{"-"*50}\n{pages[0].page_content[:50]}')

 Loaded 4 pages from "sample_knowledge.txt"

 Sample Page [0] metadata : {'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'sample_knowledge.txt', 'file_path': 'sample_knowledge.txt', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'Innomatics Research Labs', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}

 Sample Page [0] content  :
--------------------------------------------------
Innomatics Research Labs – Organizational Overview


##  Step 2 — Split into Chunks with LangChain

In [7]:
# RecursiveCharacterTextSplitter splits on: paragraphs → sentences → words
# It preserves LangChain Document structure + keeps original metadata (page, source)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(pages)

print(f' Created {len(chunks)} chunks from {len(pages)} pages')
print(f' Avg chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars')
print(f'\n Sample Chunk [0]:')
print(f'   Metadata : {chunks[0].metadata}')
print(f'   Content  :\n{"-"*50}\n{chunks[0].page_content}')

 Created 108 chunks from 4 pages
 Avg chunk length: 80 chars

 Sample Chunk [0]:
   Metadata : {'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'sample_knowledge.txt', 'file_path': 'sample_knowledge.txt', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'Innomatics Research Labs', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}
   Content  :
--------------------------------------------------
Innomatics Research Labs – Organizational Overview and Key Personnel


##  Step 3 — Load HuggingFace Embedding Model via LangChain

In [8]:
print(f' Loading embedding model: {EMBEDDING_MODEL} ...')

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},   # change to 'cuda' if you have a GPU
    encode_kwargs={"normalize_embeddings": True}  # normalise for cosine similarity
)

# Quick sanity check
test_vector = embedding_model.embed_query("Hello, LangChain!")
print(f' Model loaded! Embedding dimension: {len(test_vector)}')

 Loading embedding model: paraphrase-MiniLM-L3-v2 ...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\DELL\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--paraphrase-MiniLM-L3-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Model loaded! Embedding dimension: 384


## 7️⃣ Step 4 — Store Chunks + Embeddings in ChromaDB via LangChain

In [ ]:
# Chroma.from_documents() does everything in one call:
#   1. Embeds all chunks using the provided embedding model
#   2. Stores vectors + text + metadata in ChromaDB
#   3. Persists the DB to CHROMA_DIR on disk

print(' Embedding chunks and storing in ChromaDB...')

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR
)

print(f' Successfully stored {len(chunks)} chunks in ChromaDB!')
print(f' Persisted at: {CHROMA_DIR}')

##  Step 5 — Semantic Search using LangChain Retriever

In [ ]:
# ── Similarity Search (with scores) ──────────────────────────────────────────
QUERY = "What is this document about?"   #  Change this to your question
TOP_K = 3

results = vectorstore.similarity_search_with_score(QUERY, k=TOP_K)

print(f' Query: "{QUERY}"')
print('=' * 65)

for i, (doc, score) in enumerate(results):
    print(f'\n Result #{i+1}')
    print(f'   Similarity Score : {score:.4f}  (lower = more similar for L2)')
    print(f'   Source           : {doc.metadata.get("source", "N/A")}')
    print(f'   Page             : {doc.metadata.get("page", "N/A")}')
    print(f'   Content Preview  :\n{"-"*65}\n{doc.page_content[:400]}')

##  Step 6 — Use as LangChain Retriever (for RAG pipelines)

In [ ]:
# Convert vectorstore into a LangChain Retriever
# This is the standard interface used in RAG (Retrieval-Augmented Generation) chains

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

retrieved_docs = retriever.invoke(QUERY)

print(f' Retrieved {len(retrieved_docs)} docs via LangChain Retriever')
for i, doc in enumerate(retrieved_docs):
    print(f'\n[{i+1}] Page {doc.metadata.get("page", "?")} — {doc.page_content[:200]}...')

##  (Optional) Reload Existing ChromaDB in a New Session

In [ ]:
# Run this cell in a fresh session — no need to re-embed!

embedding_model_reload = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore_reload = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
    embedding_function=embedding_model_reload
)

print(f' Reloaded ChromaDB from "{CHROMA_DIR}"')
print(f' Documents in collection: {vectorstore_reload._collection.count()}')

##  Step 7 — Connect Retriever to LLM (Basic RAG Chain)
Now we take our `retriever` and connect it to a free Groq LLM to actually answer questions based on the PDF context!

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize LLM
os.environ['GROQ_API_KEY'] = 'gsk_YOUR_API_KEY_HERE'
llm = ChatGroq(model='llama3-8b-8192')

# 2. Create Prompt Template
template = """
You are a helpful assistant. Answer the question based ONLY on the following context.
If the answer is not in the context, say "I don't know based on the document."

Context: {context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

# 3. Format Context Helper Function
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Build the RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Test the Chain
answer = rag_chain.invoke("What is this document about?")
print(" AI Answer:")
print("-" * 50)
print(answer)

---
##  Full Pipeline Summary

```
PDF File
   │
   ▼
PyMuPDFLoader          → List of LangChain Documents (1 per page)
   │
   ▼
RecursiveCharacterTextSplitter  → Smaller overlapping chunk Documents
   │
   ▼
HuggingFaceEmbeddings  → Dense vector for each chunk (384-dim)
   │
   ▼
Chroma (ChromaDB)      → Persisted vector store on disk
   │
   ▼
VectorStoreRetriever   → Plug into any LangChain RAG chain
```

| Setting | Value |
|---|---|
| Chunk size | 100 chars |
| Chunk overlap | 20 chars |
| Embedding model | sentence-transformers/all-MiniLM-L6-v2 (384-dim) |
| Vector DB | ChromaDB (local, persisted) |
| Similarity metric | Cosine |

>  **Next step:** Plug the `retriever` into a `RetrievalQA` or `ConversationalRetrievalChain` to build a full RAG Q&A chatbot over your PDF!